In [ ]:
from google.colab import drive
import os
drive.mount('/content/gdrive')
os.chdir("gdrive/MyDrive/Manuscripts/Livingness")

**Figure 2 plot:** Subjective Ratings of Likability, Perceived Livingness, and Perceived Complexity (perceived-livingness means on 53 images; see Data exclusions)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------------
# Figure 2: subjective ratings, computed from the participant-level data
# ------------------------------------------------------------------
# Perceived-livingness responses for seven images were recorded as a constant
# value across participants (response-recording fault; see Methods, Data
# exclusions). They are set to missing here and excluded from all
# livingness means. Likability and complexity use all 60 images.
EXCLUDED_LIVINGNESS = ['A6b', 'A10b', 'B3a', 'B8b', 'C2a', 'C5a', 'C9a']

df = pd.read_csv("Data/full_anonymized.csv")
for img in EXCLUDED_LIVINGNESS:
    df[img + '2'] = np.nan          # column suffix 2 = perceived livingness item

RATINGS = {'Likability': '1', 'Perceived Livingness': '2', 'Perceived Complexity': '3'}
GROUPS  = {'A': 'Geometric patterns', 'B': 'Architectural details', 'C': 'Building facades'}
LEVELS  = {'a': 'High L', 'b': 'Low L'}

def cell_mean(group, level, suffix):
    cols = [c for c in df.columns
            if c.startswith(group) and c.endswith(level + suffix)
            and c[len(group):-2].isdigit()]
    return df[cols].mean(axis=1, skipna=True)   # per-participant mean over the images in the cell

group_labels, colors, hatch_patterns = [], [], []
palette = {'A': '#EE6677', 'B': '#228833', 'C': '#4477AA'}   # Tol bright
for g in 'ABC':
    for lev in 'ab':
        group_labels.append(f"{GROUPS[g]} {LEVELS[lev]}")
        colors.append(palette[g]); hatch_patterns.append('' if lev == 'a' else '//')
alpha_values = [0.85] * 6

metrics = list(RATINGS.keys())
data, error_data = [], []
for m, sfx in RATINGS.items():
    vals, sems = [], []
    for g in 'ABC':
        for lev in 'ab':
            v = cell_mean(g, lev, sfx)
            vals.append(v.mean()); sems.append(v.sem())
    data.append(vals); error_data.append(sems)

# Print the plotted values so they can be checked against Table S1/S2
for m, vals, sems in zip(metrics, data, error_data):
    print(m, [f"{v:.3f}±{s:.3f}" for v, s in zip(vals, sems)])

# ------------------------------------------------------------------ plot
fig, ax = plt.subplots(figsize=(12, 6))
bar_width = 0.12
x = np.arange(len(metrics))
for i in range(len(group_labels)):
    offset = (i - 2.5) * bar_width
    values = [data[j][i] for j in range(len(metrics))]
    errors = [error_data[j][i] for j in range(len(metrics))]
    ax.bar(x + offset, values, bar_width, yerr=errors, capsize=5,
           color=colors[i], alpha=alpha_values[i],
           edgecolor='black', linewidth=0.8,
           label=group_labels[i], hatch=hatch_patterns[i])

ax.set_ylabel('Mean Rating', fontsize=14)
ax.set_title('Subjective Ratings of Likability, Perceived Livingness, and Perceived Complexity',
             fontsize=20, pad=20)
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=14)
ax.set_ylim(0, 7)

handles, labels = ax.get_legend_handles_labels()
unique = dict(zip(labels, handles))
ax.legend(unique.values(), unique.keys(),
          loc='lower center', bbox_to_anchor=(0.5, -0.25),
          ncol=3, fontsize=12, frameon=False)

plt.tight_layout()
plt.savefig('./Figure2_Final.pdf', bbox_inches='tight', facecolor='white')
plt.savefig('./Figure2_Final.png', dpi=400, bbox_inches='tight', facecolor='white')
plt.show()


**Figure 3 plot:** Percentage of Participants Preferring High Livingness

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

# Data for Figure 3
metrics = ['Likability', 'Livingness']
group_labels = ['Geometric patterns', 'Architectural details', 'Building facades']

# Tol "bright" colorblind-safe palette — SAME as Figure 2
TOL_RED   = '#EE6677'  # Geometric patterns
TOL_GREEN = '#228833'  # Architectural details
TOL_BLUE  = '#4477AA'  # Building facades

colors = {'Geometric patterns':   TOL_RED,
          'Architectural details': TOL_GREEN,
          'Building facades':      TOL_BLUE}
alpha_values = [0.85, 0.85, 0.85, 0.85, 0.85, 0.85]
# alpha_values = [0.1, 0.1, 0.1, 0.1, 0.1, 0.1]

# Values: per group, per metric
likability = [0.7746, 0.85079, 0.87619]
livingness = [0.78889, 0.82381, 0.85079]
data = [likability, livingness]

likability_sem = [0.025896, 0.027423, 0.018893]
livingness_sem = [0.030536, 0.036114, 0.020349]
error_data = [likability_sem, livingness_sem]

# Bar settings
bar_width = 0.2
x = np.arange(len(metrics))

# Plot setup
fig, ax = plt.subplots(figsize=(8, 6))

# Plot bars per group
bars = []  # to collect bar objects for adding labels
for i, group in enumerate(group_labels):
    group_values = [data[j][i] for j in range(len(metrics))]
    group_errors = [error_data[j][i] for j in range(len(metrics))]
    offset = (i - 1) * bar_width  # centers the bars nicely

    bars_group = ax.bar(x + offset, group_values, bar_width,
                        yerr=group_errors, capsize=5,
                        color=colors[group], edgecolor='black', linewidth=0.8,
                        label=group, alpha=alpha_values[i])
    bars.append(bars_group)

# Labels and formatting
ax.set_title('Percentage of Participants Preferring High Livingness',
             fontsize=16, pad=20)
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=14)

# Y-axis from 0.7 to 0.9 → format as percentage
# ax.set_ylim(0.7, 0.9)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_ylim(0.5, 1.0)
ax.axhline(0.5, color='#444444', linestyle='--', linewidth=1.2, alpha=0.9, zorder=1)
ax.text(-0.55, 0.505, 'chance (50%)',
        ha='left', va='bottom',
        fontsize=10, color='#444444', style='italic')

# Add numeric labels on top of each bar
for bars_group in bars:
    for bar in bars_group:
        height = bar.get_height()
        ax.annotate(f'{height*100:.1f}%',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(-20, 3),
                    textcoords='offset points',
                    ha='center', va='bottom', fontsize=11)

# Legend below the figure
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, labels,
          loc='lower center', bbox_to_anchor=(0.5, -0.25),
          ncol=3, fontsize=12, frameon=False)

# Adjust layout to make space for legend
plt.tight_layout(rect=[0, 0.1, 1, 1])

# --- Export ---
# Vector PDF (best for print, scalable, small file)
plt.savefig('Figure3_Final.pdf', bbox_inches='tight', facecolor='white')

# 400 DPI PNG (raster, for portals that prefer images)
plt.savefig('Figure3_Final.png', dpi=400, bbox_inches='tight', facecolor='white')

plt.show()

**Figure 4 plot:** Aesthetic Emotions in Response to High vs. Low Livingness Images

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Emotion categories
categories = [
    "Aversive Emotions", "Sadness", "Prototypical Aesthetic Emotions",
    "Epistemic Emotions", "Animation", "Nostalgia/Relaxation", "Amusement"
]

# Mean emotion ratings
high_l  = [1.54603, 1.37302, 2.31746, 2.45238, 2.24074, 2.42063, 2.59921]
low_l = [1.75238, 1.53968, 1.73333, 1.80423, 1.61640, 2.15873, 1.82143]

# SEM values
sem_high_l  = [0.056562, 0.063970, 0.080148, 0.093635, 0.102715, 0.065197, 0.077421]
sem_low_l = [0.082643, 0.092126, 0.065270, 0.085047, 0.080414, 0.077421, 0.077421]

# Tol "bright" colorblind-safe palette — consistent with Figures 2 and 3
TOL_BLUE = '#4477AA'   # High L
TOL_GREY = '#BBBBBB'   # Low L (Tol's neutral grey)

TOL_PURPLE = '#AA3377'


# Bar positions
x = np.arange(len(categories))
width = 0.35

# Plot setup
fig, ax = plt.subplots(figsize=(12, 8))

# IMPORTANT: bars1 and bars2 labels were swapped in the original code.
# bars1 (left bar, x - width/2) plotted high_l but was labelled 'Low L'.
# Fixed below: left bar = High L (colored), right bar = Low L (grey).
bars1 = ax.bar(x - width/2, low_l, width,
               label='Low L',
               color=TOL_GREY, edgecolor='black', linewidth=0.8, hatch='//',
               yerr=sem_low_l, capsize=5, alpha=1)

bars2 = ax.bar(x + width/2, high_l, width,
               label='High L',
               color=TOL_PURPLE, edgecolor='black', linewidth=0.8,
               yerr=sem_high_l, capsize=5, alpha=1)

# Highlight aversive/sadness categories (negative emotions)
highlight_indices = [0, 1]
x_start = x[highlight_indices[0]] - width
x_end = x[highlight_indices[-1]] + width
ax.axvspan(x_start - 0.2, x_end + 0.2,
           color='lightblue', alpha=0.35, zorder=0,
           label='Negative Emotions')

# Axis labels and ticks
ax.set_ylabel('Mean Emotion Rating', fontsize=14)
ax.set_title('Aesthetic Emotions in Response to High vs. Low Livingness Images',
             fontsize=20, pad=20)
ax.set_xticks(x)
ax.set_xticklabels(categories, rotation=30, ha='right', fontsize=13)
ax.set_ylim(0, 3.2)

# Add data labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.2f}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 10),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=11)

# Legend (centered below the plot)
ax.legend(loc='lower center', bbox_to_anchor=(0.5, -0.35),
          ncol=3, frameon=False, fontsize=12)

# Layout and save
plt.tight_layout(rect=[0, 0.05, 1, 1])

# --- Export ---
plt.savefig('Figure4_Final.pdf', bbox_inches='tight', facecolor='white')
plt.savefig('Figure4_Final.png', dpi=400, bbox_inches='tight', facecolor='white')

plt.show()

**High vs. Low livingness**

In [ ]:
import pandas as pd
import numpy as np

# Rebuild the per-participant aggregates used by Figure 5 from the participant-level
# data, applying the perceived-livingness exclusion (see Figure 2 cell).
EXCLUDED_LIVINGNESS = ['A6b', 'A10b', 'B3a', 'B8b', 'C2a', 'C5a', 'C9a']
df = pd.read_csv("Data/full_anonymized.csv")
for img in EXCLUDED_LIVINGNESS:
    df[img + '2'] = np.nan

def agg(suffix, level):
    cols = [c for c in df.columns if c[0] in 'ABC' and c.endswith(level + suffix) and c[1:-2].isdigit()]
    return df[cols].mean(axis=1, skipna=True)

likability_df = pd.DataFrame({
    "Connectedness to Nature": df["Q3Ave"],
    "High L": agg('1', 'a'), "Low L": agg('1', 'b')})
likability_df["High vs. Low L"] = likability_df["High L"] - likability_df["Low L"]

livingness_df = pd.DataFrame({
    "Connectedness to Nature": df["Q3Ave"],
    "High L": agg('2', 'a'), "Low L": agg('2', 'b')})
livingness_df["High vs. Low L"] = livingness_df["High L"] - livingness_df["Low L"]

# Overwrite the derived CSVs so the repo copies match the figure
likability_df.round(3).to_csv("Data/likability.csv", index=False)
livingness_df.round(3).to_csv("Data/livingness.csv", index=False)
print(likability_df.head()); print(livingness_df.head())


**Figure 5 plot:** Relationship between Connectedness to Nature and Subjective Ratings

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import linregress

# Re-load the uploaded CSV files
likability_df = pd.read_csv("Data/likability.csv")
livingness_df = pd.read_csv("Data/livingness.csv")

# Extract data
cn_like  = likability_df["Connectedness to Nature"]
high_l_like = likability_df["High L"]
low_l_like  = likability_df["Low L"]
diff_like   = likability_df["High vs. Low L"]

cn_live  = livingness_df["Connectedness to Nature"]
high_l_live = livingness_df["High L"]
low_l_live  = livingness_df["Low L"]
diff_live   = livingness_df["High vs. Low L"]

# Tol "bright" colorblind-safe palette — consistent with previous figures
TOL_PURPLE = '#AA3377'   # High L  (matches Figure 4 "High L" purple)
TOL_GREY   = '#777777'   # Low L   (slightly darker than Figure 4's bar grey for scatter visibility)
TOL_YELLOW = '#CCBB44'   # Difference

# Prepare side-by-side subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7), sharey=True)
ax1.axhline(0, color='gray', linestyle=':', linewidth=1, alpha=0.5, zorder=0)
ax2.axhline(0, color='gray', linestyle=':', linewidth=1, alpha=0.5, zorder=0)

def plot_panel(ax, cn, high_l, low_l, diff, title, show_legend_labels=True):
    """Helper to plot one panel."""
    # Scatter points
    ax.scatter(cn, high_l, color=TOL_PURPLE, s=50, alpha=0.7,
               edgecolor='black', linewidth=0.5,
               label='High L' if show_legend_labels else None)
    ax.scatter(cn, low_l,  color=TOL_GREY,   s=50, alpha=0.7,
               edgecolor='black', linewidth=0.5,
               label='Low L' if show_legend_labels else None)
    ax.scatter(cn, diff,   color=TOL_YELLOW, s=50, alpha=0.7,
               edgecolor='black', linewidth=0.5,
               label='High vs. Low L' if show_legend_labels else None)

    # Regression lines
    x_vals = np.linspace(cn.min(), cn.max(), 100)
    for y, color, lbl in [(high_l, TOL_PURPLE, 'Linear (High L)'),
                          (low_l,  TOL_GREY,   'Linear (Low L)'),
                          (diff,   TOL_YELLOW, 'Linear (High vs. Low)')]:
        reg = linregress(cn, y)
        ax.plot(x_vals, reg.intercept + reg.slope * x_vals,
                color=color, linewidth=2,
                label=lbl if show_legend_labels else None)

    ax.set_xlabel('Connectedness to Nature', fontsize=20)
    ax.set_title(title, fontsize=22, pad=15)
    ax.tick_params(axis='both', labelsize=16)
    ax.set_xlim(2, 5)


# Plot both panels
plot_panel(ax1, cn_like, high_l_like, low_l_like, diff_like,
           'Likability', show_legend_labels=True)
plot_panel(ax2, cn_live, high_l_live, low_l_live, diff_live,
           'Perceived Livingness', show_legend_labels=False)

ax1.set_ylabel('Ratings', fontsize=20)
ax1.set_ylim(-1, 9)

# Shared legend below the figure
handles, labels = ax1.get_legend_handles_labels()
fig.legend(handles, labels,
           loc='lower center', bbox_to_anchor=(0.5, -0.06),
           ncol=3, fontsize=18, frameon=False)

# Suptitle
plt.suptitle('Relationship between Connectedness to Nature and Subjective Ratings',
             fontsize=26)

plt.tight_layout(rect=[0, 0.12, 1, 0.96])

# --- Export ---
plt.savefig('Figure5_Final.pdf', bbox_inches='tight', facecolor='white')
plt.savefig('Figure5_Final.png', dpi=400, bbox_inches='tight', facecolor='white')

plt.show()